After running classify_scans.py:
1. Align all sources 
2. Detect duplicated scans 
3. Detect misaligned labeling

In [ ]:
import pandas as pd
import os

# load all csv files in the current directory into a single DataFrame

# path to all csv files
path = "/home/gaia/Projects/legacy_data/manual_labeling/" # UPDATE PATH AS NEEDED
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
dataframes = [pd.read_csv(os.path.join(path, f)) for f in csv_files]
combined_label_df = pd.concat(dataframes, ignore_index=True)


In [4]:
print(f"Total rows in combined DataFrame: {len(combined_label_df)}")

#unique scans = combination of subject_id and session_id
print(f"amount of unique scans: {len(combined_label_df[['subject_id', 'session_id']].drop_duplicates())}")

Total rows in combined DataFrame: 847
amount of unique scans: 833


In [5]:
duplicates = combined_label_df[combined_label_df.duplicated(subset=['subject_id', 'session_id'], keep=False)]

# order by subject_id and session_id for easier inspection
duplicates = duplicates.sort_values(by=['subject_id', 'session_id'])
print(f"Number of duplicate rows based on subject_id and session_id: {len(duplicates)}")

# for every duplicated subject_id and session_id, check if classification_label is the same
conflicting_labels = duplicates.groupby(['subject_id', 'session_id']).filter(lambda x: len(x['classification_label'].unique()) > 1)
print(f"Number of rows with conflicting labels for the same subject_id and session_id: {len(conflicting_labels)}")

Number of duplicate rows based on subject_id and session_id: 28
Number of rows with conflicting labels for the same subject_id and session_id: 18


go over "conflicting labels" and manually and align 

### temporary resolution of conflicts 

In [6]:
# temporary - for each scan (subject_id, session_id), get the higher classification_label

# in conflicting_labels, for each (subject_id, session_id), get the max classification_label
resolved_labels = conflicting_labels.groupby(['subject_id', 'session_id'])['classification_label'].max().reset_index()

# apply these resolved labels back to the combined_label_df
for index, row in resolved_labels.iterrows():
    combined_label_df.loc[
        (combined_label_df['subject_id'] == row['subject_id']) & 
        (combined_label_df['session_id'] == row['session_id']), 
        'classification_label'
    ] = row['classification_label']

# make sure there are no more conflicting labels
duplicates = combined_label_df[combined_label_df.duplicated(subset=['subject_id', 'session_id'], keep=False)]
conflicting_labels = duplicates.groupby(['subject_id', 'session_id']).filter(lambda x: len(x['classification_label'].unique()) > 1)
print(f"Number of rows with conflicting labels after resolution: {len(conflicting_labels)}")

Number of rows with conflicting labels after resolution: 0


In [7]:
combined_label_df = combined_label_df.sort_values(by=['subject_id', 'session_id'])

# add the manual label column to combined_df by matching the subject_id and session_id
combined_df = pd.read_pickle('/home/gaia/Projects/legacy_data/combined_gm_volumes.pkl')
print(combined_df.columns)


Index(['subject_id', 'session_id', 'region_label', 'tissue_type', 'volume_mm3',
       'tiv', 'sex', 'institute', 'manufacturer', 'age_in_years', 'dob',
       'gm_volume_cm3', 'protocol', 'source', 'birth_year', 'Unnamed: 0',
       'atlas_name', 'scan_time', 'age_at_scan', 'weight', 'directory_path',
       'estimated_critical_info', 'scan_date', 'file_path'],
      dtype='object')


In [8]:
# remove 'file_path' column from combined_label_df 
combined_label_df = combined_label_df[['subject_id', 'session_id', 'classification_label']]

In [9]:
# merge combined_label_df to combined_df based on matching subject_id and session_id 
combined_df = combined_df.merge(
    combined_label_df, 
    on=['subject_id', 'session_id'], 
    how='left'
)

print(combined_df.columns)



Index(['subject_id', 'session_id', 'region_label', 'tissue_type', 'volume_mm3',
       'tiv', 'sex', 'institute', 'manufacturer', 'age_in_years', 'dob',
       'gm_volume_cm3', 'protocol', 'source', 'birth_year', 'Unnamed: 0',
       'atlas_name', 'scan_time', 'age_at_scan', 'weight', 'directory_path',
       'estimated_critical_info', 'scan_date', 'file_path',
       'classification_label'],
      dtype='object')


In [10]:
combined_df.to_pickle('/home/gaia/Projects/legacy_data/combined_gm_volumes.pkl')